# AA MD Simulations for CG Force Field Parameterization

Run all-atom MD on multiple molecules using Colab's free T4 GPU.
Downloads trajectory files for CG force field parameterization.

**Runtime → Change runtime type → T4 GPU**

In [ ]:
# Install dependencies
!pip install -q openmm
!pip install -q git+https://github.com/WestonVoglesonger/adaptive-cg.git

import openmm
print(f"OpenMM {openmm.__version__}")
for i in range(openmm.Platform.getNumPlatforms()):
    print(f"  Platform: {openmm.Platform.getPlatform(i).getName()}")

In [ ]:
# Fetch molecules
!acg fetch --category proteins
!acg fetch --category nucleic_acids
!acg list

In [ ]:
# Molecules to simulate — representative set covering different types/sizes
MOLECULES = [
    "1CRN",   # crambin, 46 residues (tiny protein)
    "1UBQ",   # ubiquitin, 76 residues (small protein)
    "1L2Y",   # trp-cage, 20 residues (miniprotein)
    "1LYZ",   # lysozyme, 129 residues (medium protein)
    "1BNA",   # B-DNA dodecamer (nucleic acid)
    "1IGD",   # protein G, 61 residues
    "1PGA",   # protein G variant
    "1MBN",   # myoglobin, 153 residues (larger)
]

# Simulation settings
PRODUCTION_STEPS = 500_000   # 1 ns
EQUILIBRATION = 25_000       # 50 ps
SAVE_INTERVAL = 1000         # every 2 ps → 500 frames

In [ ]:
import time
from pathlib import Path
from adaptive_cg.core.simulation import SimulationConfig, run_aa_simulation

config = SimulationConfig(
    production_steps=PRODUCTION_STEPS,
    equilibration_steps=EQUILIBRATION,
    save_interval=SAVE_INTERVAL,
)

results = {}
for mol_id in MOLECULES:
    pdb_path = Path(f"data/structures/{mol_id}.pdb")
    if not pdb_path.exists():
        print(f"\n*** SKIPPING {mol_id} — PDB not found ***\n")
        continue

    output_dir = Path(f"data/trajectories/{mol_id}")
    print(f"\n{'='*60}")
    print(f"  {mol_id}")
    print(f"{'='*60}")

    t0 = time.time()
    try:
        result = run_aa_simulation(
            pdb_path=pdb_path,
            output_dir=output_dir,
            config=config,
            save_forces=True,
            verbose=True,
        )
        elapsed = time.time() - t0
        results[mol_id] = {"status": "success", "time": elapsed, **result}
        print(f"\n  Completed in {elapsed:.0f}s")
    except Exception as e:
        elapsed = time.time() - t0
        results[mol_id] = {"status": "failed", "error": str(e), "time": elapsed}
        print(f"\n  FAILED after {elapsed:.0f}s: {e}")

print(f"\n\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
for mol_id, r in results.items():
    status = r['status']
    t = r['time']
    if status == 'success':
        print(f"  {mol_id}: {t:.0f}s, {r['n_frames']} frames, {r['n_atoms']} atoms")
    else:
        print(f"  {mol_id}: FAILED — {r['error']}")

In [ ]:
# Zip all trajectories for download
import shutil
shutil.make_archive("trajectories", "zip", "data/trajectories")
print("Created trajectories.zip")
print(f"Size: {Path('trajectories.zip').stat().st_size / 1e6:.1f} MB")

# Download (works in Colab)
try:
    from google.colab import files
    files.download("trajectories.zip")
except ImportError:
    print("Not in Colab — download trajectories.zip manually")